In [4]:
import pandas as pd
import statsmodels.api as sm
from turtle import pd


import numpy as np
import datetime
import numpy as np
import pandas as pd

import plotly.graph_objects as go
from ipywidgets import widgets
from IPython.display import display
from scipy import stats
from scipy.stats import linregress
from sklearn.preprocessing import StandardScaler
from statsmodels.multivariate.manova import MANOVA
import matplotlib.pyplot as plt

df = pd.read_csv("data/all_devices_summary_5.csv")

# Fix amplifier column name
df = df.rename(columns={"Amplifier_board": "Amplifier_Board"})

# Create numeric columns for all metrics
# Yield as numeric (0–100)
df["yield_numeric"] = (
    df["Yield"]
    .astype(str)
    .str.rstrip("%")
    .replace("", "0")
    .astype(float)
)

# Blind_Ch_Imp_numeric this column contains values like "1.2 MΩ" or "500 kΩ". We need to extract the numeric part and convert it to a consistent unit (e.g., kΩ). The regex removes non-numeric characters except for the decimal point and negative sign, then we convert to numeric and handle units if needed (assuming all values are in kΩ or MΩ, we can convert MΩ to kΩ by multiplying by 1000 if the original string contains "M"). For simplicity, let's just extract the numeric part and convert to kΩ if "M" is present.
temp = df["Blind_Ch_Imp"].astype(str).str.replace(r'[^0-9.\-]', '', regex=True)
df["Blind_Ch_Imp_numeric"] = pd.to_numeric(temp, errors="coerce")

# Impedance numeric (integer kOhm)
df["Impedance_kOhm_numeric"] = pd.to_numeric(
    df["Impedance_kOhm"].astype(str).str.replace(r"[^0-9.]", "", regex=True),
    errors="coerce"
).round().astype("Int64")  # nullable integer

# CAR and NoCAR numeric
df["CAR_uV_numeric"] = pd.to_numeric(df["CAR_uV"], errors="coerce")
df["NoCAR_uV_numeric"] = pd.to_numeric(df["NoCAR_uV"], errors="coerce")

# Device category: A, B, or C, this line creates a new column "Device_Category" by taking the first character of the "Amplifier_Board" column, which indicates the device category (A, B, or C). This will allow us to group our data by device category in our analysis.
df["Device_Category"] = df["Amplifier_Board"].str[0]

#print("df_clean columns:", df_clean.columns.tolist())
#print("df columns:", df.columns.tolist())  # Original df might have it
#print(df_clean.head(3))  # See structure


print("Missing values check:")# this will show us how many missing values we have in each of the relevant numeric columns. It's important to check this before we proceed with any analysis, as missing values can affect our results and may need to be handled (e.g., by imputation or by dropping rows).
print(df[["CAR_uV_numeric", "yield_numeric", "Blind_Ch_Imp_numeric", "Impedance_kOhm_numeric", "NoCAR_uV_numeric"]].isnull().sum()) # this will give us a count of how many missing values we have in each of the relevant numeric columns. If we have a lot of missing values, we might need to consider how to handle them (e.g., by imputation or by dropping rows).
# df_clean is a copy of the original DataFrame but only with the relevant numeric columns for our analysis.
df_clean = df[["CAR_uV_numeric", "yield_numeric", "Blind_Ch_Imp_numeric", 
               "Impedance_kOhm_numeric", "NoCAR_uV_numeric"]].copy()
# for col in df_clean.columnns, what this does is it converts the columns to numeric, coercing errors to NaN. Then we drop any rows with NaN values, ensuring that we only keep rows with valid numeric data for our analysis. 
print(f"Rows after cleaning: {len(df_clean)}") # this will show us how many rows we have left after cleaning the data. It's important to check this because if we have too few rows, our analysis might not be reliable.
for col in df_clean.columns:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')


print("df_clean columns:", df_clean.columns.tolist())
print("df columns:", df.columns.tolist())  # Original df might have it
print(df_clean.head(3))

# this will convert any non-numeric values to NaN, which we can then drop.dropna() will remove any rows that contain NaN values in any of the specified columns, ensuring that we only keep rows with valid numeric data for our analysis.
df_clean = df_clean.dropna()
print(f"Clean shape: {df_clean.shape}")# this will show us the shape of the cleaned DataFrame, which should have no missing values and only numeric data in the relevant columns. It's important to check this to confirm that our cleaning process worked as expected and that we have a suitable dataset for our analysis.
print("Dtypes:", df_clean.dtypes)  # ALL float64! this will show us the data types of each column in the cleaned DataFrame. We expect all columns to be of type float64, which is important for our analysis. If any columns are not float64, we may need to convert them to ensure that our statistical models work correctly.



# Convert ALL to float64 (fixes Int64/int64)
df_clean["Impedance_kOhm_numeric"] = df_clean["Impedance_kOhm_numeric"].astype(float)
df_clean["Blind_Ch_Imp_numeric"] = df_clean["Blind_Ch_Imp_numeric"].astype(float)

print("Fixed dtypes:", df_clean.dtypes)  # Now ALL float64!

X=sm.add_constant(df_clean[["yield_numeric", "Blind_Ch_Imp_numeric", "Impedance_kOhm_numeric", "NoCAR_uV_numeric"]])
model=sm.GLM(df_clean["CAR_uV_numeric"], X, family=sm.families.Gamma()).fit()
model2=sm.MixedLM(df_clean["CAR_uV_numeric"], X, groups=df["Device_Category"]).fit()
print(model.summary())

for cat in df['Device_Category'].unique():
    subset = df[df['Device_Category'] == cat].dropna(subset=['CAR_uV_numeric', 'NoCAR_uV_numeric'])
    if len(subset) > 10:  # Min size
        X_sub = sm.add_constant(subset[['NoCAR_uV_numeric']])  # Simplified
        model_sub = sm.GLM(subset['CAR_uV_numeric'], X_sub, family=sm.families.Gaussian()).fit()
        print(f"\n=== {cat} (n={len(subset)}) ===")
        print(model_sub.summary())

# Creates separate slopes/intercepts per category
#model_interact = sm.MixedLM("CAR_uV_numeric ~ yield_numeric * Device_Category + Impedance_kOhm_numeric * Device_Category + NoCAR_uV_numeric * Device_Category", 
                             #data=df, groups="Device_Category").fit()
#print(model_interact.summary())

#plt.scatter(model2.fittedvalues, model2.resid_response)

Missing values check:
CAR_uV_numeric            0
yield_numeric             0
Blind_Ch_Imp_numeric      0
Impedance_kOhm_numeric    0
NoCAR_uV_numeric          0
dtype: int64
Rows after cleaning: 114
df_clean columns: ['CAR_uV_numeric', 'yield_numeric', 'Blind_Ch_Imp_numeric', 'Impedance_kOhm_numeric', 'NoCAR_uV_numeric']
df columns: ['Amplifier_Board', 'Wafer_Device_ID', 'AJP_Pattern', 'Date_Code', 'Time_Code', 'Low_Noise_Ch', 'Ch_count', 'Yield', 'Impedance_kOhm', 'CAR_uV', 'NoCAR_uV', 'Blind_Ch_Imp', 'Low_Imp_High_RMS_ch', 'High_Imp_High_RMS_ch', 'yield_numeric', 'Blind_Ch_Imp_numeric', 'Impedance_kOhm_numeric', 'CAR_uV_numeric', 'NoCAR_uV_numeric', 'Device_Category']
   CAR_uV_numeric  yield_numeric  Blind_Ch_Imp_numeric  \
0            8.98           70.0               8070236   
1            5.85           77.0               8216650   
2            3.29           89.0              10030516   

   Impedance_kOhm_numeric  NoCAR_uV_numeric  
0                     264              9.

c:\Users\jhowa\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:308: DomainWarning: The InversePower link function does not respect the domain of the Gamma family.
  warnings.warn((f"The {type(family.link).__name__} link function "
c:\Users\jhowa\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\genmod\families\family.py:143: RuntimeWarning: overflow encountered in square
  return 1. / (self.link.deriv(mu)**2 * self.variance(mu))
c:\Users\jhowa\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\genmod\families\links.py:325: RuntimeWarning: divide by zero encountered in power
  return np.power(z, 1. / self.power)
c:\Users\jhowa\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\genmod\families\family.py:775: RuntimeWarning: invalid value encountered in divide
  resid_dev = -np.log(endog_mu) + (endog - mu) / mu
c:\Users\jhowa\AppData\Local\Programs\Python\Python313\Lib\site-p

ValueError: NaN, inf or invalid value detected in weights, estimation infeasible.